In [ ]:
from flowmia import FlowMIA

## FlowMIA: Avaliando Ataques de Inferência de Membros em Modelos Generativos de Dados de Fluxo de Rede

Foram utilizados dois datasets: CIDDS-001 e TON-IoT. Ambos são datasets de fluxo de rede. O CIDDS-001 foi dividido em treino e teste e foi colocado na pasta datasets/real/cidds_train.csv, enquanto que o teste está em datasets/real/cidds_test.csv. O dataset do TON-IoT está em datasets/reference/ton.csv.

Com o cidds_train foram treinados 3 modelos generativos para dados tabulares: CTGAN, NetShare e Tabula. Na pasta datasets/synthetic estão os datasets sintéticos gerados por cada modelo. 

Será realizado um ataque de inferência de membros nesses modelos, ou seja, dados os dados sintéticos, o objetivo é descobrir amostras que foram usadas no treino. 

O atacante possui duas informações: o conjunto de dados sintéticos gerados pelo modelo e um dataset de referência de mesmo domínio (ou seja, o atacante sabe o domínio dos dados)

O dataset de teste, cidds_test, é usado unicamente para a análise de utilidade. 

Definição:

- **Membros**: cidds_train.csv: dados utilizados para treinamento dos modelos

- **Não-Membros**: ton.csv: dados de referência de mesmo domínio

- **Sintéticos**: ctgan.csv, netshare.csv, tabula.csv: dados sintéticos gerados por cada modelo

- **Teste**: cidds_test.csv: dados de teste amostrados da mesma distribuição dos membros para utilidade


Segue o arquivo de configuração padrão. Para cada modelo atacado serão ajustados o synth_path e save_path.

In [2]:
import pandas as pd

df = pd.read_csv('datasets/real/cidds_train.csv')
df.head()

,srcip,dstip,srcport,dstport,proto,ts,td,pkt,byt,label
0,3232291855,3232261125,48888,445,TCP,1.489536e+15,0.004,2,174.0,0
1,3232291855,3232261125,48888,445,TCP,1.489536e+15,0.004,2,174.0,0
2,3232261125,3232291855,445,48888,TCP,1.489536e+15,0.000,1,108.0,0
3,3232261125,3232291855,445,48888,TCP,1.489536e+15,0.000,1,108.0,0
4,3232291856,3232261125,58844,445,TCP,1.489536e+15,0.004,2,174.0,0


In [ ]:
{
    'member_path': 'datasets/real/cidds_train.csv', # path dos membros
    'non_member_path': 'datasets/reference/ton.csv', # path dos não-membros
    'synth_path': 'datasets/synthetic/netshare.csv', # path dos sintéticos
    'test_path': 'datasets/real/cidds_test.csv', # path do teste
    'categorical_cols': ['proto'], # colunas categóricas
    'numerical_cols': ['srcport', 'dstport', 'td', 'pkt', 'byt'], #colunas numéricas
    'ip_cols': ['srcip', 'dstip'], # colunas de ip
    'label_col': 'label', # nome da coluna do rótulo 
    'batch_size': 200, # número de amostrar por lote
    'num_epochs': 500, # número de épocas
    'fcheckpoint': 100, # frequência para salvar o checkpoint
    'save_path': 'results_archive/examples/netshare'    # pasta para salvar resultados
}

### NetShare

In [ ]:
config_netshare = {
    'member_path': 'datasets/real/cidds_train.csv', # path dos membros
    'non_member_path': 'datasets/reference/ton.csv', # path dos não-membros
    'synth_path': 'datasets/synthetic/netshare.csv', # path dos sintéticos
    'test_path': 'datasets/real/cidds_test.csv', # path do teste
    'categorical_cols': ['proto'], # colunas categóricas
    'numerical_cols': ['srcport', 'dstport', 'td', 'pkt', 'byt'], #colunas numéricas
    'ip_cols': ['srcip', 'dstip'], # colunas de ip
    'label_col': 'label', # nome da coluna do rótulo 
    'batch_size': 200, # número de amostrar por lote
    'num_epochs': 10, # número de épocas
    'fcheckpoint': 5, # frequência para salvar o checkpoint
    'save_path': 'results_archive/examples/netshare'    # pasta para salvar resultados
}

In [5]:
flowmia_netshare = FlowMIA(config=config_netshare) # cria um objeto de classe FlowMIA

Executa o MIA. Uma GAN é treinada com os dados sintéticos gerados pelo modelo. O pré-processador dessa GAN é ajustado com os sintéticos + não-membros, ou seja, o atacante tem o conhecimento dos sintéticos e do domínio do problema.

Essa GAN é treinada. O gerador gera amostras falsas, enquanto que o discrminador tenta distinguir essas amostras falsas das amostras sintéticas de treinamento. Ele atribui um score no intervalo de 0 a 1 para cada amostra, sendo que, quando mais perto de 1, mais parecido com os sintéticos a amostra é, enquanto que quanto mais próximo de 0, menos parecido ou mais próximo do aleatório.

Após o treinamento, o discriminador é utilizado para inferência de membros. São passados para ele, os membros, não-membros, os próprios sintéticos e também amostras ruidosas. São observados os scores de output do discriminador. 

A função retorna o histórico de treinamento (loss do gerador e loss do discrminador), além do resultado do MIA.

In [ ]:
mia_results = flowmia_netshare.flowmiagan(plot=True, pre_trained_model='results_archive/results/netshare/checkpoints/checkpoint_epoch_500.pth') 

In [ ]:
dcr_results = flowmia_netshare.compute_dcr(test_size=5000)

In [ ]:
dcr_results

In [32]:
fidelity = flowmia_netshare.evaluate_fidelity(plot=True)

In [9]:
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

classifiers = [MLPClassifier(hidden_layer_sizes=(20,10), max_iter=100, random_state=42),
                    DecisionTreeClassifier(max_depth=12, min_samples_leaf=50),
                    KNeighborsClassifier(n_neighbors=5),
                    RandomForestClassifier(max_depth=12, min_samples_leaf=50, n_estimators=100, class_weight="balanced")]

In [ ]:
utility_dict = flowmia_netshare.evaluate_utility(classifiers=classifiers, plot=True)

Utility for classifier: MLPClassifier
Running RTR evaluation...
Running TSTR evaluation...
Utility for classifier: DecisionTreeClassifier
Running RTR evaluation...
Running TSTR evaluation...
Utility for classifier: KNeighborsClassifier
Running RTR evaluation...
Running TSTR evaluation...
Utility for classifier: RandomForestClassifier
Running RTR evaluation...
Running TSTR evaluation...


### CTGAN

In [ ]:
config_ctgan = {
    'member_path': 'datasets/real/cidds_train.csv', # path dos membros
    'non_member_path': 'datasets/reference/ton.csv', # path dos não-membros
    'synth_path': 'datasets/synthetic/ctgan.csv', # path dos sintéticos
    'test_path': 'datasets/real/cidds_test.csv', # path do teste
    'categorical_cols': ['proto'], # colunas categóricas
    'numerical_cols': ['srcport', 'dstport', 'td', 'pkt', 'byt'], #colunas numéricas
    'ip_cols': ['srcip', 'dstip'], # colunas de ip
    'label_col': 'label', # nome da coluna do rótulo 
    'batch_size': 200, # número de amostrar por lote
    'num_epochs': 10, # número de épocas
    'fcheckpoint': 5, # frequência para salvar o checkpoint
    'save_path': 'results_archive/examples/ctgan'    # pasta para salvar resultados
}

In [ ]:
flowmia_ctgan = FlowMIA(config=config_ctgan) # cria um objeto de classe FlowMIA

In [ ]:
mia_results = flowmia_ctgan.flowmiagan(plot=True) 

In [ ]:
dcr_results = flowmia_ctgan.compute_dcr(test_size=5000)

In [17]:
fidelity = flowmia_ctgan.evaluate_fidelity(plot=True)

In [18]:
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

classifiers = [MLPClassifier(hidden_layer_sizes=(20,10), max_iter=100, random_state=42),
                    DecisionTreeClassifier(max_depth=12, min_samples_leaf=50),
                    KNeighborsClassifier(n_neighbors=5),
                    RandomForestClassifier(max_depth=12, min_samples_leaf=50, n_estimators=100, class_weight="balanced")]


utility_dict = flowmia_ctgan.evaluate_utility(classifiers=classifiers, plot=True)

Utility for classifier: MLPClassifier
Running RTR evaluation...
Running TSTR evaluation...
Utility for classifier: DecisionTreeClassifier
Running RTR evaluation...
Running TSTR evaluation...
Utility for classifier: KNeighborsClassifier
Running RTR evaluation...
Running TSTR evaluation...
Utility for classifier: RandomForestClassifier
Running RTR evaluation...
Running TSTR evaluation...


### Tabula

In [ ]:
config_tabula = {
    'member_path': 'datasets/real/cidds_train.csv', # path dos membros
    'non_member_path': 'datasets/reference/ton.csv', # path dos não-membros
    'synth_path': 'datasets/synthetic/tabula.csv', # path dos sintéticos
    'test_path': 'datasets/real/cidds_test.csv', # path do teste
    'categorical_cols': ['proto'], # colunas categóricas
    'numerical_cols': ['srcport', 'dstport', 'td', 'pkt', 'byt'], #colunas numéricas
    'ip_cols': ['srcip', 'dstip'], # colunas de ip
    'label_col': 'label', # nome da coluna do rótulo 
    'batch_size': 200, # número de amostrar por lote
    'num_epochs': 10, # número de épocas
    'fcheckpoint': 5, # frequência para salvar o checkpoint
    'save_path': 'results_archive/examples/tabula'    # pasta para salvar resultados
}

In [20]:
flowmia_tabula = FlowMIA(config=config_tabula) # cria um objeto de classe FlowMIA

In [ ]:
mia_results = flowmia_tabula.flowmiagan(plot=True) 

In [ ]:
dcr_results = flowmia_tabula.compute_dcr(test_size=5000)

In [23]:
fidelity = flowmia_tabula.evaluate_fidelity(plot=True)

In [24]:
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

classifiers = [MLPClassifier(hidden_layer_sizes=(20,10), max_iter=100, random_state=42),
                    DecisionTreeClassifier(max_depth=12, min_samples_leaf=50),
                    KNeighborsClassifier(n_neighbors=5),
                    RandomForestClassifier(max_depth=12, min_samples_leaf=50, n_estimators=100, class_weight="balanced")]


utility_dict = flowmia_tabula.evaluate_utility(classifiers=classifiers, plot=True)

Utility for classifier: MLPClassifier
Running RTR evaluation...
Running TSTR evaluation...
Utility for classifier: DecisionTreeClassifier
Running RTR evaluation...
Running TSTR evaluation...
Utility for classifier: KNeighborsClassifier
Running RTR evaluation...
Running TSTR evaluation...
Utility for classifier: RandomForestClassifier
Running RTR evaluation...
Running TSTR evaluation...
